# LangChain L8 — Level 7 — Knowledge: retrieval and RAG
OpsPilot does not know Meridian's refund policy. No model does; it is in the company's documents.
Putting every document into every prompt does not scale, so we **retrieve** the relevant parts.

```text
Indexing (once)                       Query time (every question)
Documents                             Question
   |  split into chunks                  |  embed
   v                                     v
Chunks  --embed-->  Vector store  <--nearest chunks--  Retriever
                                         |
                                         v
                                   Model + chunks  ->  grounded answer
```

Two architectures use the same retriever:

- **2-step RAG:** retrieve, then answer. Predictable, cheap, ideal for "answer from the handbook".
- **Agentic RAG:** the agent owns a `search_policies` tool and decides *whether* and *how often*
  to search. Better when a question needs several lookups or reasoning between them.

In [ ]:
%pip install -q -U langchain-huggingface sentence-transformers

### Step 1 — Documents, chunks, embeddings, vector store, retriever

The policy documents are created inline. `RecursiveCharacterTextSplitter` cuts them into
chunks that fit a prompt; an **embedding model** turns each chunk into a vector so that
"money back after 45 days" lands near the refund-window sentence even though no words match.
The embedding model runs locally (a 90 MB download); if it is unavailable, a keyword embedding
keeps the section runnable.

In [ ]:
from langchain_core.documents import Document                    # LangChain: text + metadata
from langchain_core.embeddings import Embeddings                  # LangChain: base class for embedding models
from langchain_core.vectorstores import InMemoryVectorStore       # LangChain: a vector store in RAM
from langchain_text_splitters import RecursiveCharacterTextSplitter   # LangChain: chunking

POLICY_DOCS = {                                                   # ours: Meridian's three policy documents
    "refund_policy.md": """Meridian Supply Co. Refund Policy.
Standard and Pro customers may request a refund within 30 days of purchase.
Enterprise customers may request a refund within 60 days of purchase.
Refunds above 1,000 USD require approval from a finance manager.
Duplicate charges are refunded in full at any time, regardless of the purchase date.
Refunds are returned to the original payment method within 5 business days.""",
    "shipping_policy.md": """Meridian Supply Co. Shipping Policy.
Standard shipping takes 5 to 7 business days. Express shipping takes 2 business days.
Orders above 2,000 USD ship free. Shipping to remote areas may add 3 business days.
Damaged deliveries must be reported within 48 hours with a photo.""",
    "escalation_policy.md": """Meridian Supply Co. Escalation Policy.
High-priority tickets receive a first response within 4 hours.
Enterprise customers have a dedicated account manager who must be copied on refunds.
Any action that moves money requires a human approval step.""",
}

documents = [Document(page_content=text, metadata={"source": name}) for name, text in POLICY_DOCS.items()]
splitter = RecursiveCharacterTextSplitter(chunk_size=160, chunk_overlap=20)
chunks = splitter.split_documents(documents)
print(f"{len(documents)} documents -> {len(chunks)} chunks; first chunk: {chunks[0].page_content[:80]!r} from {chunks[0].metadata['source']}")

class KeywordEmbeddings(Embeddings):                              # ours, on LangChain's Embeddings interface
    """Fallback: a hashed bag-of-words vector. Matches on shared words only, but needs no download."""
    def _embed(self, text):
        vector = [0.0] * 256
        for word in re.findall(r"[a-z]+", text.lower()):
            vector[hash(word) % 256] += 1.0
        return vector
    def embed_documents(self, texts): return [self._embed(t) for t in texts]
    def embed_query(self, text): return self._embed(text)

try:
    from langchain_huggingface import HuggingFaceEmbeddings        # LangChain integration: local sentence-transformers model
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    print("embeddings : all-MiniLM-L6-v2 (semantic)")
except Exception as exc:
    embeddings = KeywordEmbeddings()
    print("embeddings : keyword fallback (", type(exc).__name__, ")")

vector_store = InMemoryVectorStore.from_documents(chunks, embeddings)   # LangChain: embed and index every chunk
retriever = vector_store.as_retriever(search_kwargs={"k": 3})           # LangChain: "give me the 3 nearest chunks"

for doc in retriever.invoke("How long do enterprise customers have to get their money back?"):
    print(" -", doc.metadata["source"], "|", doc.page_content[:90].replace("\n", " "))

### Step 2 — 2-step RAG: retrieve, then answer

No agent, no loop: a fixed pipeline. The prompt tells the model to answer **only** from the
supplied chunks and to say so when they do not contain the answer. That instruction is what
turns retrieval into *grounded* answering.

In [ ]:
def answer_from_policies(question: str) -> str:                              # ours: the whole 2-step pipeline
    docs = retriever.invoke(question)                                        # step 1: retrieve (LangChain retriever)
    context = "\n\n".join(f"[{d.metadata['source']}] {d.page_content}" for d in docs)
    reply = model.invoke([                                                   # step 2: answer
        SystemMessage("Answer ONLY from the policy excerpts below. If they do not contain the answer, say so.\n\n" + context),
        HumanMessage(question),
    ])
    return text_of(reply)

print(answer_from_policies("A Pro customer bought a router 45 days ago and wants a refund. Is that allowed?"))

### Step 3 — Agentic RAG: retrieval as a tool

Wrap the retriever in a tool and give it to OpsPilot. Now the agent searches when it judges
that policy matters, can search more than once, and can combine policy with CRM data. The
question below needs the customer's plan **and** the refund window for that plan.

In [ ]:
@tool                                                                        # LangChain decorator, our body
def search_policies(query: str) -> str:
    """Search Meridian's policy documents (refunds, shipping, escalation). Returns the most relevant excerpts with sources."""
    docs = retriever.invoke(query)
    return "\n\n".join(f"[{d.metadata['source']}] {d.page_content}" for d in docs)

KNOWLEDGE_TOOLS = READ_TOOLS + [search_policies]
opspilot_rag = create_agent(
    model=model, tools=KNOWLEDGE_TOOLS,
    system_prompt=OPSPILOT_PROMPT + " For any question about policy, search the policy documents and cite the source file.",
)
result = opspilot_rag.invoke({"messages": [{"role": "user", "content": "Customer C001 bought order O1001 12 days ago and wants a refund. Does the refund policy allow it for their plan?"}]})
show_messages(result["messages"])

### Recap

- **Problem seen:** the model had no way to know company policy.
- **Layer added:** loader -> splitter -> embeddings -> vector store -> retriever, used as a pipeline (2-step RAG) or as a tool (agentic RAG).
- **Evidence:** the retriever returned refund chunks for a question that shared no words with them; the agent combined CRM data and policy.